In [ ]:
!pip install polars==1.39.3 fastexcel[polars]==0.19.0 xlsxwriter==3.2.0

# Завантажуємо Excel файл експортований з НІТ



1.   Експортуйте один або кілька файлів з оцінками з НІТ
2.   Завантажте всі файли одночасно



In [ ]:
from google.colab import files

uploaded = files.upload()
scores_file_paths = list(uploaded.keys())

if not scores_file_paths:
    raise ValueError('Не завантажено жодного файлу.')

print(f'Завантажено файлів: {len(scores_file_paths)}')
for i, path in enumerate(scores_file_paths, start=1):
    print(f'{i}. {path}')


In [ ]:
import polars as pl

In [ ]:
if 'scores_file_paths' not in globals() or not scores_file_paths:
    raise NameError("'scores_file_paths' is not defined or empty. Run the upload cell first.")

# Excel row 3 contains criterion markers (1, 2, 3, 4)
criteria_markers_row_index = 2

def parse_criterion(value):
    if value is None:
        return None
    text = str(value).strip()
    if text.endswith('.0'):
        text = text[:-2]
    if text in {'1', '2', '3', '4'}:
        return int(text)
    return None

results_by_file = {}
criteria_by_file = {}

for scores_file_path in scores_file_paths:
    raw = pl.read_excel(scores_file_path, has_header=False)
    name_col = raw.columns[1]  # Column B in Excel

    criteria_columns = {i: [] for i in range(1, 5)}
    marker_row = raw.row(criteria_markers_row_index, named=True)

    for col_name, marker_value in marker_row.items():
        criterion = parse_criterion(marker_value)
        if criterion is not None:
            criteria_columns[criterion].append(col_name)

    all_score_columns = [c for cols in criteria_columns.values() for c in cols]

    # Student data starts after row 3 (0-based index => from row 4)
    students = raw.slice(criteria_markers_row_index + 1)
    name_expr = pl.col(name_col).cast(pl.Utf8).str.strip_chars()
    students = students.filter(name_expr.is_not_null() & (name_expr != ''))

    # Cast score columns to numeric; non-numeric values become null
    students = students.with_columns(
        [pl.col(c).cast(pl.Float64, strict=False) for c in all_score_columns]
        if all_score_columns
        else []
    )

    agg_exprs = []
    avg_columns = []

    for criterion in range(1, 5):
        cols = criteria_columns[criterion]
        count_col = f'Кількість оцінок {criterion}'
        avg_col = f'Середня {criterion}'
        avg_columns.append(avg_col)

        if cols:
            agg_exprs.append(
                pl.sum_horizontal([pl.col(c).is_not_null().cast(pl.Int64) for c in cols]).alias(count_col)
            )
            agg_exprs.append(pl.mean_horizontal([pl.col(c) for c in cols]).alias(avg_col))
        else:
            agg_exprs.append(pl.lit(0).alias(count_col))
            agg_exprs.append(pl.lit(None).cast(pl.Float64).alias(avg_col))

    overall_avg_col = 'Семестрова'

    result = (
        students.select([
            pl.col(name_col).alias('Учень'),
            *agg_exprs,
        ])
        # Семестрова = (Середня 1 + Середня 2 + Середня 3 + Середня 4) / 4
        # Missing averages are treated as 0 in this formula.
        .with_columns(
            (
                pl.sum_horizontal([pl.col(c).fill_null(0.0) for c in avg_columns]) / 4
            ).alias(overall_avg_col)
        )
        .with_columns([pl.col(c).round(2) for c in [*avg_columns, overall_avg_col]])
        .sort('Учень')
    )

    results_by_file[scores_file_path] = result
    criteria_by_file[scores_file_path] = criteria_columns

# Keep the first file result in `result` for convenience/preview in notebook
first_file = scores_file_paths[0]
result = results_by_file[first_file]

print(f'Опрацьовано файлів: {len(scores_file_paths)}')
for file_name in scores_file_paths:
    print(f'\nФайл: {file_name}')
    print('Detected score columns by criterion:')
    for criterion, cols in criteria_by_file[file_name].items():
        print(f'Criterion {criterion}: {len(cols)} columns -> {cols}')
    print(f'Кількість учнів у звіті: {results_by_file[file_name].height}')

result

# Збережіть окремий файл з результатом для кожного завантаженого Excel.

In [ ]:
# Export results to Excel (without pandas), apply formatting, and download one ZIP in Google Colab
if 'results_by_file' not in globals() or not results_by_file:
    raise NameError("'results_by_file' is not defined or empty. Run the processing cell first.")

import os
import re
import zipfile
import xlsxwriter
from xlsxwriter.utility import xl_col_to_name

from google.colab import files

exported_files = []
used_output_names = set()

for source_path, result_df in results_by_file.items():
    # Build output file name from original uploaded file name
    source_file_name = os.path.basename(source_path)
    source_stem = os.path.splitext(source_file_name)[0]

    class_match = re.search(r'\b\d{1,2}-[A-Za-zА-Яа-яІіЇїЄєҐґ]\b', source_stem)
    group_match = re.search(r'група\s*\d+', source_stem, flags=re.IGNORECASE)

    class_name = class_match.group(0) if class_match else 'Невідомий клас'
    group_name = group_match.group(0) if group_match else 'група ?'

    # Normalize group capitalization to: "група X"
    group_name = re.sub(r'^група', 'група', group_name, flags=re.IGNORECASE)

    base_name = f'Зведені оцінки {class_name} - {group_name}'

    # Make file name unique to avoid overwriting when multiple files share class/group
    output_file = f'{base_name}.xlsx'
    if output_file in used_output_names:
        safe_source_stem = re.sub(r'[^\w\s\-А-Яа-яІіЇїЄєҐґ]', '', source_stem).strip()
        safe_source_stem = re.sub(r'\s+', ' ', safe_source_stem) or 'файл'
        output_file = f'{base_name} ({safe_source_stem}).xlsx'

        suffix = 2
        while output_file in used_output_names:
            output_file = f'{base_name} ({safe_source_stem})_{suffix}.xlsx'
            suffix += 1

    used_output_names.add(output_file)

    headers = result_df.columns
    rows = result_df.to_dicts()

    workbook = xlsxwriter.Workbook(output_file)
    worksheet = workbook.add_worksheet('Результати')
    red_format = workbook.add_format({'bg_color': '#FFC7CE', 'font_color': '#9C0006'})

    # Write header row
    for col_idx, col_name in enumerate(headers):
        worksheet.write(0, col_idx, col_name)

    # Write data rows
    for row_idx, row_data in enumerate(rows, start=1):
        for col_idx, col_name in enumerate(headers):
            worksheet.write(row_idx, col_idx, row_data.get(col_name))

    # Highlight count and average cells in red when corresponding count equals 0
    if rows:
        first_data_row = 1
        last_data_row = len(rows)

        for criterion in range(1, 5):
            count_col_name = f'Кількість оцінок {criterion}'
            avg_col_name = f'Середня {criterion}'

            if count_col_name in headers and avg_col_name in headers:
                count_col_idx = headers.index(count_col_name)
                avg_col_idx = headers.index(avg_col_name)

                count_col_letter = xl_col_to_name(count_col_idx)
                formula = f'=${count_col_letter}2=0'

                worksheet.conditional_format(
                    first_data_row, count_col_idx, last_data_row, count_col_idx,
                    {'type': 'formula', 'criteria': formula, 'format': red_format}
                )
                worksheet.conditional_format(
                    first_data_row, avg_col_idx, last_data_row, avg_col_idx,
                    {'type': 'formula', 'criteria': formula, 'format': red_format}
                )

    # Improve readability with auto-like column widths
    for col_idx, col_name in enumerate(headers):
        max_len = len(str(col_name))
        for row_data in rows:
            value = row_data.get(col_name)
            value_len = len('' if value is None else str(value))
            if value_len > max_len:
                max_len = value_len
        worksheet.set_column(col_idx, col_idx, min(max_len + 2, 45))

    workbook.close()
    exported_files.append(output_file)

zip_name = 'Зведені оцінки - всі файли.zip'
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for file_name in exported_files:
        zipf.write(file_name, arcname=file_name)

print('Експортовано файли:')
for file_name in exported_files:
    print(f'- {file_name}')

print(f'\nСтворено архів: {zip_name}')
files.download(zip_name)